# Movie Box Office Prediction

## 2. Feature Engineering, Model Training, and Evaluation

In this notebook, we'll transform our lists of genres and cast into numerical features using encoding techniques, split the data, and train models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import ast

import warnings
warnings.filterwarnings('ignore')

### Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/cleaned_movie_dataset.csv')
# Convert stringified lists back to actual python lists
df['genres_list'] = df['genres_list'].apply(ast.literal_eval)
df['cast_list'] = df['cast_list'].apply(ast.literal_eval)
df.head(2)

### Feature Engineering
We need to create numerical features out of our categorical lists. We'll use MultiLabelBinarizer for `genres_list` to create one-hot encoded columns for each genre.

In [ ]:
mlb = MultiLabelBinarizer()
genre_encoded = pd.DataFrame(mlb.fit_transform(df['genres_list']), columns=mlb.classes_, index=df.index)
df_features = pd.concat([df[['budget']], genre_encoded], axis=1)

# Let's also create a feature for cast size
df_features['cast_size'] = df['cast_list'].apply(len)

X = df_features
y = df['revenue']
X.head()

### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

### Model 1: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Random Forest Metrics:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_preds)))
print("MAE:", mean_absolute_error(y_test, rf_preds))
print("R2 Score:", r2_score(y_test, rf_preds))

### Model 2: XGBoost Regressor

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("XGBoost Metrics:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, xgb_preds)))
print("MAE:", mean_absolute_error(y_test, xgb_preds))
print("R2 Score:", r2_score(y_test, xgb_preds))

### Save the Model & Artifacts

In [ ]:
# Determine the better model; let's say XGBoost is our chosen model
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(xgb_model, '../models/xgboost_model.pkl')
joblib.dump(mlb.classes_, '../models/genre_classes.pkl')
print("Saved model and classes to models/ directory.")